# 02 - Data Preprocessing: Cleaning Your Data

Welcome to Notebook 02! In this lesson, we'll learn:
1. How to handle missing values
2. How to deal with outliers
3. How to scale/normalize features
4. Why preprocessing is crucial for ML models

## Why Data Preprocessing?

Raw data is often messy:
- ❌ Contains missing values
- ❌ Has outliers and errors
- ❌ Features have different scales
- ❌ May have duplicate records

**Clean data = Better model performance!**

## Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler, MinMaxScaler

sns.set_style('whitegrid')
print("Libraries imported successfully!")

## Step 2: Create Sample Data with Issues

In [ ]:
# Create messy data with missing values and outliers
np.random.seed(42)

data = {
    'Age': [25, 30, np.nan, 45, 35, 28, 999, 32, 40, 27],
    'Salary': [50000, 60000, 55000, np.nan, 75000, 48000, 52000, 58000, 90000, 51000],
    'Experience': [2, 5, 3, np.nan, 8, 3, 15, 4, 10, 2]
}

df_raw = pd.DataFrame(data)
print("Raw Data (with issues):")
print(df_raw)
print(f"\nDataset shape: {df_raw.shape}")

## Step 3: Identify Missing Values

In [ ]:
# Check missing values
print("Missing Values Count:")
print(df_raw.isnull().sum())

print("\nMissing Values Percentage:")
missing_percent = (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
print(missing_percent)

# Visualize missing values
plt.figure(figsize=(8, 4))
sns.heatmap(df_raw.isnull(), cbar=True, yticklabels=False)
plt.title('Missing Values Heatmap (White = Missing)')
plt.tight_layout()
plt.show()

## Step 4: Handle Missing Values

There are several strategies:
1. **Drop**: Remove rows/columns with missing values
2. **Mean/Median**: Fill with average value
3. **Forward Fill**: Use previous value
4. **Interpolation**: Estimate based on surrounding values

In [ ]:
# Method 1: Drop rows with missing values
df_dropped = df_raw.dropna()
print("After dropping missing values:")
print(f"Shape: {df_dropped.shape}")
print(f"Rows removed: {len(df_raw) - len(df_dropped)}")
print(df_dropped)

# ⚠️ Warning: We lost data!

In [ ]:
# Method 2: Fill with mean (better approach)
df_filled = df_raw.copy()
df_filled['Age'] = df_filled['Age'].fillna(df_filled['Age'].mean())
df_filled['Salary'] = df_filled['Salary'].fillna(df_filled['Salary'].mean())
df_filled['Experience'] = df_filled['Experience'].fillna(df_filled['Experience'].median())

print("After filling with mean/median:")
print(df_filled)
print(f"\nMissing values remaining: {df_filled.isnull().sum().sum()}")

## Step 5: Detect and Handle Outliers

In [ ]:
# Visualize outliers
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for idx, col in enumerate(df_filled.columns):
    axes[idx].boxplot(df_filled[col])
    axes[idx].set_title(f'Boxplot of {col}')
    axes[idx].set_ylabel('Value')

plt.tight_layout()
plt.show()

print("Box plot shows outliers as points beyond the whiskers.")

In [ ]:
# Method 1: Identify outliers using IQR (Interquartile Range)
def detect_outliers_iqr(data, column):
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = data[(data[column] < lower_bound) | (data[column] > upper_bound)]
    return outliers, lower_bound, upper_bound

# Detect outliers in Age
age_outliers, age_lower, age_upper = detect_outliers_iqr(df_filled, 'Age')
print("Age Outliers (IQR method):")
print(age_outliers)
print(f"Normal range: {age_lower:.2f} to {age_upper:.2f}")

In [ ]:
# Method 2: Handle outliers
# Option A: Remove outliers
df_no_outliers = df_filled[~((df_filled['Age'] < age_lower) | (df_filled['Age'] > age_upper))]
print(f"After removing outliers: {len(df_no_outliers)} rows (removed {len(df_filled) - len(df_no_outliers)})")

# Option B: Cap outliers (clip to upper/lower bounds)
df_capped = df_filled.copy()
df_capped['Age'] = df_capped['Age'].clip(lower=age_lower, upper=age_upper)
print(f"After capping outliers:")
print(df_capped)

## Step 6: Feature Scaling/Normalization

Many ML algorithms work better when features are on the same scale.

In [ ]:
# Show the problem: Different scales
print("Original Data (Different Scales):")
print(df_capped.describe())

# Note: Age is 20-45, Salary is 48000-90000, Experience is 2-15
# This difference in scale can affect model training!

In [ ]:
# Method 1: Standardization (Z-score normalization)
# Formula: (x - mean) / std_dev
# Result: Mean=0, Std=1

scaler_standard = StandardScaler()
df_standardized = pd.DataFrame(
    scaler_standard.fit_transform(df_capped),
    columns=df_capped.columns
)

print("After Standardization:")
print(df_standardized.describe())
print("\nNow all features have mean ≈ 0 and std ≈ 1")

In [ ]:
# Method 2: Min-Max Normalization
# Formula: (x - min) / (max - min)
# Result: Values between 0 and 1

scaler_minmax = MinMaxScaler()
df_normalized = pd.DataFrame(
    scaler_minmax.fit_transform(df_capped),
    columns=df_capped.columns
)

print("After Min-Max Normalization:")
print(df_normalized.describe())
print("\nNow all features are scaled between 0 and 1")

## Step 7: Visualize Before and After

In [ ]:
# Compare before and after
fig, axes = plt.subplots(3, 3, figsize=(15, 12))

for idx, col in enumerate(df_capped.columns):
    # Original
    axes[idx, 0].hist(df_capped[col], bins=10, edgecolor='black', alpha=0.7, color='blue')
    axes[idx, 0].set_title(f'{col} - Original')
    axes[idx, 0].set_ylabel('Frequency')
    
    # Standardized
    axes[idx, 1].hist(df_standardized[col], bins=10, edgecolor='black', alpha=0.7, color='green')
    axes[idx, 1].set_title(f'{col} - Standardized')
    axes[idx, 1].set_ylabel('Frequency')
    
    # Normalized
    axes[idx, 2].hist(df_normalized[col], bins=10, edgecolor='black', alpha=0.7, color='orange')
    axes[idx, 2].set_title(f'{col} - Normalized')
    axes[idx, 2].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

## Step 8: Complete Preprocessing Pipeline

In [ ]:
def preprocess_data(df, method='standardize'):
    """
    Complete preprocessing pipeline
    
    Steps:
    1. Handle missing values
    2. Handle outliers
    3. Scale features
    """
    
    df_processed = df.copy()
    
    # Step 1: Handle missing values
    for column in df_processed.columns:
        if df_processed[column].isnull().sum() > 0:
            df_processed[column].fillna(df_processed[column].mean(), inplace=True)
    
    # Step 2: Handle outliers (IQR method)
    for column in df_processed.columns:
        Q1 = df_processed[column].quantile(0.25)
        Q3 = df_processed[column].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        df_processed[column] = df_processed[column].clip(lower=lower, upper=upper)
    
    # Step 3: Scale features
    if method == 'standardize':
        scaler = StandardScaler()
    else:
        scaler = MinMaxScaler()
    
    df_processed = pd.DataFrame(
        scaler.fit_transform(df_processed),
        columns=df_processed.columns
    )
    
    return df_processed, scaler

# Apply preprocessing
df_processed, scaler = preprocess_data(df_raw, method='standardize')

print("Final Preprocessed Data:")
print(df_processed)
print(f"\nShape: {df_processed.shape}")
print(f"Missing values: {df_processed.isnull().sum().sum()}")

## Key Takeaways 🎯

1. **Missing Values**: Fill or drop (depending on how many)
2. **Outliers**: Detect with IQR, then remove or cap
3. **Scaling**: Use StandardScaler or MinMaxScaler
4. **Order Matters**: 
   - Missing values → Outliers → Scaling
5. **Why Scaling?**: 
   - Some algorithms are sensitive to scale (KNN, SVM)
   - Makes training faster
   - Prevents features with larger scale from dominating

## When to Use What?

| Task | Method | When to Use |
|------|--------|-------------|
| Missing Values | Mean/Median | Small percentage of missing data |
| Missing Values | Drop | >30% missing or small dataset |
| Outliers | Remove | Few outliers, have enough data |
| Outliers | Cap | Want to keep all data |
| Scaling | StandardScaler | Distance-based algorithms (KNN, SVM, K-means) |
| Scaling | MinMaxScaler | Neural networks, want bounded values |
| Scaling | No scaling | Tree-based algorithms (Decision Trees, Random Forests) |

## Next Steps

Now that your data is clean, move to **Notebook 03** to learn **Feature Engineering**!